In [1]:
! pip install open3d

  Using cached configargparse-1.7.1-py3-none-any.whl.metadata (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 39.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 36.2 MB/s eta 0:00:00 0:00:01
Using cached configargparse-1.7.1-py3-none-any.whl (25 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 37.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 35.0 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def load_obj(path):
    """Load vertices and triangular faces from a Wavefront OBJ file."""
    vertices = []
    faces = []
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'{path} does not exist')
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            if line.startswith('v '):
                parts = line.strip().split()
                vertices.append([float(parts[1]), float(parts[2]), float(parts[3])])
            elif line.startswith('f '):
                # faces can be defined as f v, f v/vt, or f v/vt/vn
                entries = line.strip().split()[1:]
                face = []
                for entry in entries:
                    face.append(int(entry.split('/')[0]) - 1)
                if len(face) == 3:
                    faces.append(face)
                else:
                    # triangulate polygons with >3 vertices
                    for idx in range(1, len(face) - 1):
                        faces.append([face[0], face[idx], face[idx + 1]])
    if not vertices or not faces:
        raise ValueError(f'Failed to load geometry from {path}')
    return np.asarray(vertices, dtype=np.float32), np.asarray(faces, dtype=np.int32)

def show_obj(path, *, figsize=(6, 6), facecolor=(0.3, 0.6, 0.9, 0.75), edgecolor='k'):
    """Render an OBJ mesh inline using matplotlib."""
    vertices, faces = load_obj(path)
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')
    mesh = Poly3DCollection(vertices[faces], facecolor=facecolor, edgecolor=edgecolor, linewidths=0.2)
    ax.add_collection3d(mesh)
    ax.scatter(vertices[:, 0], vertices[:, 1], vertices[:, 2], s=2, c='black', alpha=0.3)

    mins = vertices.min(axis=0)
    maxs = vertices.max(axis=0)
    center = (mins + maxs) / 2.0
    span = np.max(maxs - mins) / 2.0
    span = max(span, 1e-6)
    ax.set_xlim(center[0] - span, center[0] + span)
    ax.set_ylim(center[1] - span, center[1] + span)
    ax.set_zlim(center[2] - span, center[2] + span)
    ax.set_box_aspect([1, 1, 1])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.view_init(elev=10, azim=120)
    plt.tight_layout()
    return fig, ax

# Example usage:
# fig, ax = show_obj('output/example.obj')
# fig
